# Introduction

**Text mining** is the process of extracting meaningful information, patterns, and knowledge from unstructured textual data. It combines techniques from natural language processing (NLP), machine learning, and information retrieval to analyze large volumes of text and uncover hidden insights. Text mining enables tasks such as document classification, topic modeling, keyword extraction, and sentiment analysis.

In this module, we will explore key text mining techniques through the practical lens of **sentiment analysis**—the process of identifying and categorizing emotions or opinions expressed in text. By applying methods like text preprocessing, tokenization, vectorization (e.g., TF-IDF), and classification algorithms, students will learn how to transform raw textual data into structured insights and build models capable of understanding human sentiment.

**Data Set**

​The ***Twitter US Airline Sentiment Dataset*** comprises tweets from February 2015, where travelers expressed their opinions about major U.S. airlines. Each tweet is labeled with a sentiment—positive, neutral, or negative—and includes information such as the airline name, reason for negative sentiment (if applicable), and other metadata. This dataset is commonly used for sentiment analysis and text mining tasks.​

# 01 Loading Data

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
df = pd.read_csv('/content/drive/MyDrive/CSE-Data-Mining-Warehouse-Lab-426/Week 04-Text Mining/Tweets.csv')
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


# 02 Pre-processing

For sentiment analysis, we only need two features. The text and the sentiment.

In [30]:
df = df[['text','airline_sentiment']]
df.head()

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [31]:
df

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative
...,...,...
14635,@AmericanAir thank you we got on a different f...,positive
14636,@AmericanAir leaving over 20 minutes Late Flig...,negative
14637,@AmericanAir Please bring American Airlines to...,neutral
14638,"@AmericanAir you have my money, you change my ...",negative


We need to clean the tweets to remove noise like URLs, mentions, punctuation, etc. For that, we will use the regular expression library of python. Regular expression is used to find patterns within strings. We will also take everything into lower case.

In [32]:
import re

def textCleaning(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # Remove URLs
    text = re.sub(r'@\w+|\#', '', text)  # Remove mentions and hashtags
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    return text

df['text'] = df['text'].apply(textCleaning)
df.head()

,text,airline_sentiment
0,what said,neutral
1,plus youve added commercials to the experienc...,positive
2,i didnt today must mean i need to take anothe...,neutral
3,its really aggressive to blast obnoxious ente...,negative
4,and its a really big bad thing about it,negative


**Tokenization**
Tokenization is the process of splitting text into smaller units called tokens, typically words. It helps machines process and understand language more easily by breaking down a sentence into its meaningful components. For example, consider this tweet from the dataset:

"@united I missed my flight because of a delay. Not happy!"

After tokenization (and removing mentions and punctuation), it becomes:

['I', 'missed', 'my', 'flight', 'because', 'of', 'a', 'delay', 'Not', 'happy']

These tokens are then used to extract features, compute frequencies, or feed into machine learning models. We will use the nltk library for tokenization.

In [33]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

We will perform the following tasks for tokenization.

1. Break the sentence down into words.
2. Remove stopwords. Stop words carry less meaningful information for text analysis—such as "the", "is", "and", "in", etc.
3. Stemming. Stemming reduces words to their root form. 'Flying' turns into 'fli'.
4. Generate bigrams. Bigrams are sequences of two words like "Not Flying".

In [34]:
nltk.download('punkt_tab')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def tokenize(text):
    tokens = nltk.word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)

df['text'] = df['text'].apply(tokenize)
df.head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,text,airline_sentiment
0,said,neutral
1,plu youv ad commerci experi tacki,positive
2,didnt today must mean need take anoth trip,neutral
3,realli aggress blast obnoxi entertain guest fa...,negative
4,realli big bad thing,negative


# 03 Train-test-split

We know what it is.

In [35]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['airline_sentiment']
X_train_text, X_test_text, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [36]:
X_train_text.head()

,text
750,offer us 8 room 32 peopl fail
6875,jfk nyc staff amaz lax jetblu send email detai...
7598,well last updat right direct least ill keep fi...
14124,flight 3056 still sit dfw wait baggag load
6187,companion pass broken today purchaseerrorinval...


# 04 Feature Extraction

We'll convert the cleaned text into numerical form using TF-IDF vectorization.
1. **Term Frequency (TF)** measures how frequently a term (word) appears in a document.
2. **Inverse Document Frequency (IDF)** gives less importance to common words and more importance to rare ones

max_features is set as 3000. This will only use the top 3000 words and remove everything else.

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=3000)
X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()

In [38]:
# total size of vocublary

print(len(vectorizer.get_feature_names_out()))

3000


# 05 Classification

This step is as usual. We train a classifier on the data. We will use multiple classifiers and see which one gives us the best accuracy.

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score

# Initialize classifiers with random_state=42
classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'SVM': SVC(random_state=42),
    #'Naive Bayes': MultinomialNB(),
    #'Decision Tree': DecisionTreeClassifier(random_state=42),
    # 'Random Forest': RandomForestClassifier(random_state=42),
    #'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    #'K-Nearest Neighbors': KNeighborsClassifier()
}

results = {}
for name, model in classifiers.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    print(f"{name}: {accuracy}")

Logistic Regression: 0.7937158469945356
SVM: 0.7916666666666666


In [40]:
# prompt: train using logistic regression

# Assuming X_train, y_train, X_test, y_test are already defined from the previous code

# Initialize and train the Logistic Regression model
logreg_model = LogisticRegression(random_state=42)
logreg_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_logreg = logreg_model.predict(X_test)

# Evaluate the model (example using accuracy)
accuracy_logreg = accuracy_score(y_test, y_pred_logreg)
print(f"Logistic Regression Accuracy: {accuracy_logreg}")

Logistic Regression Accuracy: 0.7937158469945356


In [41]:
# Function to classify a single sentence
def classify_sentence(sentence, model, vectorizer):
    cleaned_sentence = textCleaning(sentence)
    tokenized_sentence = tokenize(cleaned_sentence)
    vectorized_sentence = vectorizer.transform([tokenized_sentence])
    prediction = model.predict(vectorized_sentence)[0]
    return prediction

# Example sentence to classify
sentence = "On 12 November 2024 I flew from Dhaka to Kathmundu on a smooth ontime evening flight. With a specious half-full plane. enjoyed a beautiful sunset and stunning views of himalayas and mount everest."

classify_sentence(sentence, logreg_model, vectorizer)

'positive'